In [144]:
import torch
import torch.nn as nn


In [145]:
# Data setup
input_seq = "i love programming language"
chars = list(set(input_seq))  # Unique characters
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for ch, i in char2idx.items()}
print(chars)
print(char2idx)

['l', 'i', 'o', 'g', 'm', 'v', 'n', 'e', 'a', 'u', ' ', 'p', 'r']
{'l': 0, 'i': 1, 'o': 2, 'g': 3, 'm': 4, 'v': 5, 'n': 6, 'e': 7, 'a': 8, 'u': 9, ' ': 10, 'p': 11, 'r': 12}


In [146]:
# Convert string to indices
input_data = [char2idx[ch] for ch in input_seq[:-1]]  # "hell"
target_data = [char2idx[ch] for ch in input_seq[1:]]  # "ello"

# input_data  = [1, 0, 2, 2]  # 'h', 'e', 'l', 'l'
# target_data = [0, 2, 2, 3]  # 'e', 'l', 'l', 'o'

print(input_data)
print(target_data)


[1, 10, 0, 2, 5, 7, 10, 11, 12, 2, 3, 12, 8, 4, 4, 1, 6, 3, 10, 0, 8, 6, 3, 9, 8, 3]
[10, 0, 2, 5, 7, 10, 11, 12, 2, 3, 12, 8, 4, 4, 1, 6, 3, 10, 0, 8, 6, 3, 9, 8, 3, 7]


In [ ]:
# Convert to tensors
input_tensor = torch.tensor(input_data).unsqueeze(1)  # shape: (seq_len, batch_size)
target_tensor = torch.tensor(target_data)
    
print(input_tensor)
print(input_tensor.shape)
print(target_data)

tensor([[ 1],
        [10],
        [ 0],
        [ 2],
        [ 5],
        [ 7],
        [10],
        [11],
        [12],
        [ 2],
        [ 3],
        [12],
        [ 8],
        [ 4],
        [ 4],
        [ 1],
        [ 6],
        [ 3],
        [10],
        [ 0],
        [ 8],
        [ 6],
        [ 3],
        [ 9],
        [ 8],
        [ 3]])
torch.Size([26, 1])
[10, 0, 2, 5, 7, 10, 11, 12, 2, 3, 12, 8, 4, 4, 1, 6, 3, 10, 0, 8, 6, 3, 9, 8, 3, 7]


In [148]:
# Hyperparameters
input_size = len(chars)
hidden_size = 8
output_size = len(chars)
learning_rate = 0.01
n_epochs = 100

In [149]:
# One-hot encoding
def one_hot(index, size):
    vec = torch.zeros(size)
    vec[index] = 1.0
    return vec

input_one_hot = torch.stack([one_hot(i, input_size) for i in input_data])  # (seq_len, input_size)

In [150]:
# RNN Model
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

In [151]:
model = SimpleRNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [152]:
# Training loop
for epoch in range(n_epochs):
    hidden = torch.zeros(1, 1, hidden_size)  # (num_layers, batch_size, hidden_size)
    optimizer.zero_grad()
    
    input_seq_tensor = input_one_hot.unsqueeze(1)  # shape: (seq_len, batch, input_size)
    output, hidden = model(input_seq_tensor, hidden)
    
    loss = criterion(output.squeeze(1), target_tensor)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        predicted = output.argmax(dim=2).squeeze()
        predicted_chars = ''.join([idx2char[idx.item()] for idx in predicted])
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {loss.item():.4f}, Predicted: {predicted_chars}')

Epoch [10/100], Loss: 2.3456, Predicted: gaaaggagggggggagggaagggagg
Epoch [20/100], Loss: 2.0437, Predicted: gaaaraaraarageaaggaeaggagr
Epoch [30/100], Loss: 1.6884, Predicted: ggovr aragrageagg aeng agr
Epoch [40/100], Loss: 1.3369, Predicted: glove pragrageigg lmng agr
Epoch [50/100], Loss: 0.9956, Predicted:  love pragramming long agr
Epoch [60/100], Loss: 0.7272, Predicted:  love pragramming languagr
Epoch [70/100], Loss: 0.5388, Predicted:  love programming languagr
Epoch [80/100], Loss: 0.4060, Predicted:  love programming language
Epoch [90/100], Loss: 0.3075, Predicted:  love programming language
Epoch [100/100], Loss: 0.2311, Predicted:  love programming language
